<a href="https://colab.research.google.com/github/jabri62018/Zx_Mother_Function_Jabri/blob/Zx_Mother_Function_Jabri/Zx_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:

import numpy as np
import pandas as pd

# =========================
# 1. البيانات
# =========================
gammas = np.array([
    14.13472514, 21.02203964, 25.01085758, 30.42487613, 32.93506159,
    37.58617816, 40.91871901, 43.32707328, 48.00515088, 49.77383248,
    52.97032148, 56.44624770, 59.34704400, 60.83177802, 65.11254405,
    67.07981053, 69.54640171, 72.06715767, 75.70469070, 77.14484007
])

CONSTANTS = {
    'pi': np.pi, 'e': np.e, 'phi': (1 + np.sqrt(5)) / 2,
    '1/2': 0.5, 'sqrt2': np.sqrt(2), 'sqrt3': np.sqrt(3),
    'c': 299792458.0, 'G': 6.674e-11
}

# بيانات تجريبية - بدّلها ببياناتك الحقيقية
t = np.linspace(0, 100, 2000)
Z = np.sin(np.outer(t, gammas/10)) * np.exp(-t[:, None]/50)

# =========================
# 2. الدالة
# =========================
def compute_C_and_derivs(gamma, t, z):
    z = np.array(z, dtype=np.complex128)

    if np.all(np.abs(z) < 1e-15):
        return np.nan, np.nan, np.nan, np.nan, np.nan

    dz = np.gradient(z, t)
    d2z = np.gradient(dz, t)
    d3z = np.gradient(d2z, t)

    z_mean = np.nanmean(z)
    dz_mean = np.nanmean(dz)
    d2z_mean = np.nanmean(d2z)
    d3z_mean = np.nanmean(d3z)

    C = 0.5 * gamma**2 * np.real(np.nanmean(d3z / z))
    return C, z_mean, dz_mean, d2z_mean, d3z_mean

# =========================
# 3. اللوب مع عدّاد الأصفار
# =========================
rows = []
failed_count = 0

for i, gamma in enumerate(gammas):
    z = Z[:, i]
    C_val, z_m, dz_m, d2z_m, d3z_m = compute_C_and_derivs(gamma, t, z)

    if np.isnan(C_val) or not np.isfinite(C_val):
        failed_count += 1
        continue

    logC = np.log10(abs(C_val) + 1e-300)

    best_const, best_diff = None, np.inf
    for name, const_val in CONSTANTS.items():
        log_const = np.log10(abs(const_val) + 1e-300)
        diff = abs(logC - log_const)
        if diff < best_diff:
            best_diff, best_const = diff, name

    rows.append({
        '#': i + 1,  # رقم مسلسل 1-28
        'gamma': gamma,
        'z_mean': z_m,
        'z_prime_mean': dz_m,
        'z_double_mean': d2z_m,
        'z_triple_mean': d3z_m,
        'C_raw': C_val,
        'closest_const': best_const,
        'LogDiff': best_diff
    })

df = pd.DataFrame(rows)

# =========================
# 4. الطباعة
# =========================
print("="*130)
print(" ZX MODEL - FULL DEBUG TABLE ".center(130))
print("="*130)

if df.empty:
    print(f"\n[ERROR] ما في نتائج صالحة. عدد الفاشلين: {failed_count} من {len(gammas)}")
else:
    pd.set_option('display.float_format', '{:.6e}'.format)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 200)

    print(df.to_string(index=False))

    print("\n" + "="*130)
    print(f"النتيجة: نجح {len(df)} من أصل {len(gammas)}")
    print(f"وجدنا {failed_count} قيمة فاشلة - إما أصفار أو NaN أو Inf")
    print("="*130)

    # فلترة سريعة
    threshold = 3.0
    filtered = df[(df['LogDiff'] < threshold) & (df['closest_const']!= 'c')]
    if not filtered.empty:
        print(f"\nالنتائج القريبة LogDiff < {threshold}:")
        print(filtered[['#', 'gamma', 'closest_const', 'LogDiff']].to_string(index=False))

# =========================
# 5. الحفظ
# =========================
if not df.empty:
    df.to_csv("gamma_debug.csv", index=False)
    print("\nتم حفظ الجدول كامل في: gamma_debug.csv")

                                                   ZX MODEL - FULL DEBUG TABLE                                                    
 #        gamma                     z_mean                z_prime_mean               z_double_mean              z_triple_mean         C_raw closest_const      LogDiff
 1 1.413473e+01 8.023484e-03+0.000000e+00j  3.379549e-04+0.000000e+00j -1.603941e-02+0.000000e+00j 5.210398e-04+0.000000e+00j  4.609492e+02            pi 2.166503e+00
 2 2.102204e+01 5.377158e-03+0.000000e+00j  8.122940e-04+0.000000e+00j -2.375477e-02+0.000000e+00j 7.772629e-04+0.000000e+00j -1.422954e+03            pi 2.656041e+00
 3 2.501086e+01 3.776908e-03+0.000000e+00j -6.219449e-04+0.000000e+00j -2.392404e-02+0.000000e+00j 5.391341e-03+0.000000e+00j -2.064545e+03            pi 2.817674e+00
 4 3.042488e+01 3.685479e-03+0.000000e+00j  1.300994e-03+0.000000e+00j -3.403913e-02+0.000000e+00j 1.433027e-03+0.000000e+00j  8.193466e+03            pi 3.416318e+00
 5 3.293506e+01 3.399547e-03+0.000

/tmp/ipykernel_7679/111421184.py:42: RuntimeWarning: divide by zero encountered in divide
  C = 0.5 * gamma**2 * np.real(np.nanmean(d3z / z))
/tmp/ipykernel_7679/111421184.py:42: RuntimeWarning: invalid value encountered in divide
  C = 0.5 * gamma**2 * np.real(np.nanmean(d3z / z))


In [60]:

import numpy as np
import pandas as pd

# =========================
# 1. البيانات
# =========================
gammas = np.array([
    14.13472514, 21.02203964, 25.01085758, 30.42487613, 32.93506159,
    37.58617816, 40.91871901, 43.32707328, 48.00515088, 49.77383248,
    52.97032148, 56.44624770, 59.34704400, 60.83177802, 65.11254405,
    67.07981053, 69.54640171, 72.06715767, 75.70469070, 77.14484007
])

CONSTANTS = {
    'c': 299792458.0,
    'G': 6.67430e-11,
    'h': 6.62607015e-34,
    'hbar': 1.054571817e-34,
    'k_B': 1.380649e-23,
    'e': 1.602176634e-19,
    'epsilon_0': 8.8541878128e-12,
    'mu_0': 4 * np.pi * 1e-7,
    'm_e': 9.10938356e-31,
    'm_p': 1.67262192369e-27,
    'pi': np.pi,
    '1/epsilon_0': 1 / 8.8541878128e-12,
}

# بيانات تجريبية - بدّلها ببياناتك الحقيقية
t = np.linspace(0, 100, 2000)
Z = np.sin(np.outer(t, gammas/10)) * np.exp(-t[:, None]/50)

# =========================
# 2. الدالة
# =========================
def compute_C_and_derivs(gamma, t, z):
    z = np.array(z, dtype=np.complex128)
    if np.all(np.abs(z) < 1e-15):
        return np.nan, np.nan, np.nan, np.nan, np.nan

    dz = np.gradient(z, t)
    d2z = np.gradient(dz, t)
    d3z = np.gradient(d2z, t)

    z_mean = np.nanmean(z)
    dz_mean = np.nanmean(dz)
    d2z_mean = np.nanmean(d2z)
    d3z_mean = np.nanmean(d3z)

    C = 0.5 * gamma**2 * np.real(np.nanmean(d3z / z))
    return C, z_mean, dz_mean, d2z_mean, d3z_mean

# =========================
# 3. اللوب
# =========================
rows = []
failed_count = 0

for i, gamma in enumerate(gammas):
    z = Z[:, i]
    C_val, z_m, dz_m, d2z_m, d3z_m = compute_C_and_derivs(gamma, t, z)

    if np.isnan(C_val) or not np.isfinite(C_val):
        failed_count += 1
        continue

    logC = np.log10(abs(C_val) + 1e-300)

    best_const, best_val = None, None
    best_diff = np.inf
    for name, const_val in CONSTANTS.items():
        log_const = np.log10(abs(const_val) + 1e-300)
        diff = abs(logC - log_const)
        if diff < best_diff:
            best_diff, best_const, best_val = diff, name, const_val

    rows.append({
        'X': i + 1,
        'gamma': gamma,
        'z': z_m,
        'z\'': dz_m,
        'z"': d2z_m,
        'z\'"': d3z_m,
        'محسوب': C_val,
        'معلوم': best_val,
        'المعنى': best_const
    })

df = pd.DataFrame(rows)

# =========================
# 4. الطباعة
# =========================
pd.set_option('display.float_format', '{:.6e}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)

print(df.to_string(index=False))

# الخلاصة
total = len(gammas)
success = len(df)
print(f"\nالخلاصة = وجدنا {success} / {total} صفر")
print(f"فاشلين: {failed_count}")

# =========================
# 5. الحفظ
# =========================
if not df.empty:
    df.to_csv("gamma_table.csv", index=False)
    print("\nتم حفظ الجدول في: gamma_table.csv")

 X        gamma                          z                          z'                          z"                        z'"         محسوب        معلوم المعنى
 1 1.413473e+01 8.023484e-03+0.000000e+00j  3.379549e-04+0.000000e+00j -1.603941e-02+0.000000e+00j 5.210398e-04+0.000000e+00j  4.609492e+02 3.141593e+00     pi
 2 2.102204e+01 5.377158e-03+0.000000e+00j  8.122940e-04+0.000000e+00j -2.375477e-02+0.000000e+00j 7.772629e-04+0.000000e+00j -1.422954e+03 3.141593e+00     pi
 3 2.501086e+01 3.776908e-03+0.000000e+00j -6.219449e-04+0.000000e+00j -2.392404e-02+0.000000e+00j 5.391341e-03+0.000000e+00j -2.064545e+03 3.141593e+00     pi
 4 3.042488e+01 3.685479e-03+0.000000e+00j  1.300994e-03+0.000000e+00j -3.403913e-02+0.000000e+00j 1.433027e-03+0.000000e+00j  8.193466e+03 3.141593e+00     pi
 5 3.293506e+01 3.399547e-03+0.000000e+00j  1.394620e-03+0.000000e+00j -3.678094e-02+0.000000e+00j 1.798535e-03+0.000000e+00j -1.771060e+05 2.997925e+08      c
 6 3.758618e+01 2.469147e-03+0.000000e+0

/tmp/ipykernel_7679/2140512939.py:50: RuntimeWarning: divide by zero encountered in divide
  C = 0.5 * gamma**2 * np.real(np.nanmean(d3z / z))
/tmp/ipykernel_7679/2140512939.py:50: RuntimeWarning: invalid value encountered in divide
  C = 0.5 * gamma**2 * np.real(np.nanmean(d3z / z))
